# Data Reconciliation

This notebook reconciles three loan-related datasets into a single
loan-level modeling dataset while preserving label integrity.

Datasets:
- `loan_application`: 4,368 labeled loans (master table)
- `customer_banking_profile`: customer demographic features
- `repayment_history`: historical loan data for behavioral features

Steps:
1. Define the loan application table as the modeling universe
2. Assess customer profile coverage
3. Engineer backward-looking historical features
4. Merge all data at the loan level
5. Generate reconciliation and data quality metrics

Outputs:
- `reconciled_loan_data.csv`
- `data_reconciliation_report.csv`


In [8]:
# Import Libraries 
import pandas as pd

In [9]:
# Load the cleaned datasets
loan_application = pd.read_csv(r"..\data\interim\loan_applications.csv")
customer_profile = pd.read_csv(r"..\data\interim/customer_banking_profile.csv")
repayment_history = pd.read_csv(r"..\data\interim/repayment_history.csv")

In [10]:
# Inspect dataset
print(loan_application.info())
print(loan_application.shape)
print(customer_profile.info())
print(customer_profile.shape)
print(repayment_history.info())
print(repayment_history.shape)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4368 entries, 0 to 4367
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   customerid     4368 non-null   object 
 1   systemloanid   4368 non-null   int64  
 2   loannumber     4368 non-null   int64  
 3   approveddate   4368 non-null   object 
 4   creationdate   4368 non-null   object 
 5   loanamount     4368 non-null   float64
 6   totaldue       4368 non-null   float64
 7   termdays       4368 non-null   int64  
 8   good_bad_flag  4368 non-null   object 
dtypes: float64(2), int64(3), object(4)
memory usage: 307.3+ KB
None
(4368, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4334 entries, 0 to 4333
Data columns (total 8 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   customerid                 4334 non-null   object 
 1   bank_account_type          4334 non-null   object 
 

## Critical Reconciliation Checks

In [11]:
# customer coverage check
customers_loan = set(loan_application['customerid'])
customers_profile = set(customer_profile['customerid'])

In [13]:
# missing profiles 
missing_profiles = customers_loan - customers_profile
print(f"Customers in loans: {len(customers_loan)}")
print(f"Customers in profile: {len(customers_profile)}")
print(f"Missing profiles: {len(missing_profiles)}")

Customers in loans: 4368
Customers in profile: 4334
Missing profiles: 1099


In [16]:
# Check for customers with multiple loans
multi_loan_customers = loan_application['customerid'].value_counts()
print(f"Customers with multiple loans: {(multi_loan_customers > 1).sum()}")
print(f"\nTop repeat customers:\n{multi_loan_customers.head(10)}")

Customers with multiple loans: 0

Top repeat customers:
customerid
8a2a81a74ce8c05d014cfb32a0da1049    1
8a85886e54beabf90154c0a29ae757c0    1
8a8588f35438fe12015444567666018e    1
8a85890754145ace015429211b513e16    1
8a858970548359cc0154883481981866    1
8a8589f35451855401546b0738c42524    1
8a858e095c59b91b015c5e5cea3719bc    1
8a858e1158dc4d830158f7bde4f47ea7    1
8a858e185b4923b4015b4ae48d28646a    1
8a858e1d5cd58f9e015ceda4bdb63673    1
Name: count, dtype: int64


In [18]:
# How many loans lack customer profiles?
affected_loans = loan_application[loan_application['customerid'].isin(missing_profiles)]
print(f"Loans affected by missing profiles: {len(affected_loans)}")
print(f"Percentage of total loans: {len(affected_loans)/len(loan_application)*100:.2f}%")

Loans affected by missing profiles: 1099
Percentage of total loans: 25.16%


In [20]:
# Repayment History Reconciliation
loans_master = set(loan_application['systemloanid'])
loans_history = set(repayment_history['systemloanid'])

labeled_in_history = loans_master & loans_history
unlabeled_history = loans_history - loans_master

print(f"Labeled loans (in master): {len(labeled_in_history)}")
print(f"Unlabeled loans (need filtering): {len(unlabeled_history)}")
print(f"Master loans missing history: {len(loans_master - loans_history)}")

Labeled loans (in master): 0
Unlabeled loans (need filtering): 18183
Master loans missing history: 4368


In [21]:
# Check systemloanid ranges
print("loan_app systemloanid:")
print(f"  Min: {loan_application['systemloanid'].min()}")
print(f"  Max: {loan_application['systemloanid'].max()}")
print(f"  Sample: {loan_application['systemloanid'].head(3).tolist()}")

print("\nrepayment_history systemloanid:")
print(f"  Min: {repayment_history['systemloanid'].min()}")
print(f"  Max: {repayment_history['systemloanid'].max()}")
print(f"  Sample: {repayment_history['systemloanid'].head(3).tolist()}")

loan_app systemloanid:
  Min: 301958485
  Max: 302004050
  Sample: [301994762, 301965204, 301966580]

repayment_history systemloanid:
  Min: 301600134
  Max: 302000275
  Sample: [301682320, 301883808, 301831714]


In [ ]:
# matching by loannumber
loans_master_num = set(loan_application['loannumber'])
loans_history_num = set(repayment_history['loannumber'])

labeled_in_history = loans_master_num & loans_history_num
unlabeled_history = loans_history_num - loans_master_num

print(f"Labeled loans (by loannumber): {len(labeled_in_history)}")
print(f"Unlabeled loans: {len(unlabeled_history)}")
print(f"Master loans missing history: {len(loans_master_num - loans_history_num)}")

Labeled loans (by loannumber): 22
Unlabeled loans: 4
Master loans missing history: 1


In [23]:
# matching by customerid
customers_in_history = set(repayment_history['customerid'])
customers_matched = customers_loan & customers_in_history

print(f"Customers in loan_app: {len(customers_loan)}")
print(f"Customers in repayment_history: {len(customers_in_history)}")
print(f"Customers matched: {len(customers_matched)}")
print(f"Match rate: {len(customers_matched)/len(customers_loan)*100:.2f}%")

Customers in loan_app: 4368
Customers in repayment_history: 4359
Customers matched: 4359
Match rate: 99.79%


In [ ]:
# Compare date ranges
loan_application['approveddate'] = pd.to_datetime(loan_application['approveddate'])
repayment_history['approveddate'] = pd.to_datetime(repayment_history['approveddate'])

print("loan_app date range:")
print(f"  Earliest: {loan_application['approveddate'].min()}")
print(f"  Latest: {loan_application['approveddate'].max()}")

print("\n repayment_history date range:")
print(f"  Earliest: {repayment_history['approveddate'].min()}")
print(f"  Latest: {repayment_history['approveddate'].max()}")

loan_app date range:
  Earliest: 2017-07-01 01:35:26
  Latest: 2017-07-30 22:55:51

repayment_history date range:
  Earliest: 2016-01-15 08:53:28
  Latest: 2017-07-28 10:47:43


In [27]:
# Final Reconciliation Summary
print("DATA RECONCILIATION SUMMARY")

print("\n CUSTOMER PROFILE COVERAGE:")
print(f"   Total customers: {len(customers_loan)}")
print(f"   Missing profiles: {len(missing_profiles)} ({len(missing_profiles)/len(customers_loan)*100:.1f}%)")
print(f"   With profiles: {len(customers_loan - missing_profiles)} ({(len(customers_loan - missing_profiles))/len(customers_loan)*100:.1f}%)")

print("\n REPAYMENT HISTORY LINKAGE:")
print(f"   Customers matched: {len(customers_matched)} ({len(customers_matched)/len(customers_loan)*100:.1f}%)")
print(f"   Historical loans available: {len(repayment_history)}")
print(f"   Date range: {repayment_history['approveddate'].min().date()} to {repayment_history['approveddate'].max().date()}")

print("\n MODELING UNIVERSE:")
print(f"   Target loans (July 2017): {len(loan_application)}")
print(f"   Can use historical features: {len(customers_matched)}")
print(f"   Missing both profile + history: {len(missing_profiles - customers_in_history)}")

DATA RECONCILIATION SUMMARY

 CUSTOMER PROFILE COVERAGE:
   Total customers: 4368
   Missing profiles: 1099 (25.2%)
   With profiles: 3269 (74.8%)

 REPAYMENT HISTORY LINKAGE:
   Customers matched: 4359 (99.8%)
   Historical loans available: 18183
   Date range: 2016-01-15 to 2017-07-28

 MODELING UNIVERSE:
   Target loans (July 2017): 4368
   Can use historical features: 4359
   Missing both profile + history: 4


In [ ]:
# Detailed reconciliation report
reconciliation_report = {
    'metric': [
        'Total Target Loans',
        'Unique Customers',
        'Customers with Profile',
        'Customers Missing Profile', 
        'Profile Coverage %',
        'Customers with History',
        'History Coverage %',
        'Historical Loans Available',
        'Customers Missing Both',
        'Usable for Full Feature Set'
    ],
    'count': [
        len(loan_application),
        len(customers_loan),
        len(customers_loan - missing_profiles),
        len(missing_profiles),
        f"{(len(customers_loan - missing_profiles))/len(customers_loan)*100:.1f}%",
        len(customers_matched),
        f"{len(customers_matched)/len(customers_loan)*100:.1f}%",
        len(repayment_history),
        len(missing_profiles - customers_in_history),
        len((customers_loan - missing_profiles) & customers_matched)
    ]
}

recon_df = pd.DataFrame(reconciliation_report)
print(recon_df.to_string(index=False))

# Save report
recon_df.to_csv(r'..\data\interim\data_reconciliation_report.csv', index=False)
print("\n Report saved: data_reconciliation_report.csv")

                     metric count
         Total Target Loans  4368
           Unique Customers  4368
     Customers with Profile  3269
  Customers Missing Profile  1099
         Profile Coverage % 74.8%
     Customers with History  4359
         History Coverage % 99.8%
 Historical Loans Available 18183
     Customers Missing Both     4
Usable for Full Feature Set  3264

✓ Report saved: data_reconciliation_report.csv


In [30]:
# Create reconciled modeling dataset
# Join loan_application with customer_profile (inner join - only matched customers)
modeling_data = loan_application.merge(
    customer_profile, 
    on='customerid', 
    how='inner',
    indicator=True
)

print(f"Reconciled modeling dataset: {len(modeling_data)} loans")
print(f"Lost {len(loan_application) - len(modeling_data)} loans due to missing profiles")
print(f"\nTarget distribution:\n{modeling_data['good_bad_flag'].value_counts()}")

Reconciled modeling dataset: 3269 loans
Lost 1099 loans due to missing profiles

Target distribution:
good_bad_flag
Good    2556
Bad      713
Name: count, dtype: int64


In [33]:
# Aggregate payment_history
historical_agg = repayment_history.groupby('customerid').agg({
    'systemloanid': 'count',
    'loanamount': ['mean', 'sum'],
    'termdays': 'mean'
}).reset_index()

historical_agg.columns = ['customerid', 'hist_loan_count', 'hist_avg_amount', 
                          'hist_total_borrowed', 'hist_avg_termdays']

print(f"Aggregated history: {historical_agg.shape}")
print(historical_agg.head())

Aggregated history: (4359, 5)
                         customerid  hist_loan_count  hist_avg_amount  \
0  8a1088a0484472eb01484669e3ce4e0b                1     10000.000000   
1  8a1a1e7e4f707f8b014f797718316cad                4     17500.000000   
2  8a1a32fc49b632520149c3b8fdf85139                7     12857.142857   
3  8a1eb5ba49a682300149c3c068b806c7                8     16250.000000   
4  8a1edbf14734127f0147356fdb1b1eb2                2     10000.000000   

   hist_total_borrowed  hist_avg_termdays  
0              10000.0          15.000000  
1              70000.0          37.500000  
2              90000.0          19.285714  
3             130000.0          33.750000  
4              20000.0          22.500000  


In [35]:
# Merge loan_application + customer_profile + aggregated history
final_data = loan_application.merge(
    customer_profile,
    on='customerid',
    how='left'
).merge(
    historical_agg,
    on='customerid',
    how='left'
)

print(f"Final dataset shape: {final_data.shape}")
print(f"Rows preserved: {len(final_data)} (should be 4368)")
print(f"\nNull counts:\n{final_data.isnull().sum()}")

Final dataset shape: (4368, 20)
Rows preserved: 4368 (should be 4368)

Null counts:
customerid                      0
systemloanid                    0
loannumber                      0
approveddate                    0
creationdate                    0
loanamount                      0
totaldue                        0
termdays                        0
good_bad_flag                   0
bank_account_type            1099
longitude_gps                1099
latitude_gps                 1099
bank_name_clients            1099
employment_status_clients    1099
age                          1099
state                        1136
hist_loan_count                 9
hist_avg_amount                 9
hist_total_borrowed             9
hist_avg_termdays               9
dtype: int64


In [36]:
# Check merged dataset
print(final_data.shape)
print(final_data.info())

(4368, 20)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4368 entries, 0 to 4367
Data columns (total 20 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   customerid                 4368 non-null   object        
 1   systemloanid               4368 non-null   int64         
 2   loannumber                 4368 non-null   int64         
 3   approveddate               4368 non-null   datetime64[ns]
 4   creationdate               4368 non-null   object        
 5   loanamount                 4368 non-null   float64       
 6   totaldue                   4368 non-null   float64       
 7   termdays                   4368 non-null   int64         
 8   good_bad_flag              4368 non-null   object        
 9   bank_account_type          3269 non-null   object        
 10  longitude_gps              3269 non-null   float64       
 11  latitude_gps               3269 non-null   float64       


In [37]:
# Save the final reconciled dataset
final_data.to_csv(r'..\data\processed\reconciled_loan_data.csv', index=False)

print(" Saved: data/reconciled_loan_data.csv")
print(f"\n Final Summary:")
print(f"  Total loans: {len(final_data)}")
print(f"  With customer profile: {final_data['bank_account_type'].notna().sum()} (74.8%)")
print(f"  With history: {final_data['hist_loan_count'].notna().sum()} (99.8%)")
print(f"  Complete records (all features): {final_data.dropna().shape[0]}")

 Saved: data/reconciled_loan_data.csv

 Final Summary:
  Total loans: 4368
  With customer profile: 3269 (74.8%)
  With history: 4359 (99.8%)
  Complete records (all features): 3227


## Reconciliation Complete

Final dataset: **4,368 loans × 20 features**

Coverage:
- 74.8% of loans have customer profile attributes
- 99.8% of loans have historical behavioral features

Dataset preserved at loan level and ready for modeling.
